In [1]:
import os
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# 源模型和目标保存路径
SOURCE_MODEL_PATH = "Qwen/Qwen3-8B" 
# 新建一个专门的文件夹存放剪枝后的模型
TARGET_MODEL_PATH = "/home/bev/disk4/haoxuan/SpeculativeDecoding/model/qwen3-4b-pruned-data-driven"

# 通过数据分析得出的保留名单 (1-indexed)
keep_layers_1_indexed = [1, 2, 5, 6, 7, 9, 23, 24, 25, 27, 28, 30, 31, 32, 33, 34, 35, 36]

# 转换为 Python 认识的 0-indexed (减 1)
keep_indices = [i - 1 for i in keep_layers_1_indexed]

print(f"准备保留的层索引 (0-35): {keep_indices}")
print(f"总计保留层数: {len(keep_indices)}")

准备保留的层索引 (0-35): [0, 1, 4, 5, 6, 8, 22, 23, 24, 26, 27, 29, 30, 31, 32, 33, 34, 35]
总计保留层数: 18


In [4]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download

SOURCE_REPO = "Qwen/Qwen3-8B"

print(f"🔍 正在查找 {SOURCE_REPO} 的本地缓存绝对路径...")
# 核心魔法：直接从本地缓存目录中揪出真实路径，不联网
LOCAL_MODEL_PATH = snapshot_download(repo_id=SOURCE_REPO, local_files_only=True)

print(f"🎯 找到本地真实路径:\n{LOCAL_MODEL_PATH}\n")
print("🚀 正在从本地绝对路径加载模型和 Tokenizer")

# 直接使用绝对路径进行加载
tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_PATH, 
    trust_remote_code=True,
    local_files_only=True
)

# device_map="cpu" 保证整个加载过程在内存中完成
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="cpu", 
    trust_remote_code=True,
    local_files_only=True
)

original_layer_count = model.config.num_hidden_layers
print(f"✅ 模型完美离线加载成功！当前原始层数: {original_layer_count}")

🔍 正在查找 Qwen/Qwen3-8B 的本地缓存绝对路径...
🎯 找到本地真实路径:
/home/bev/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218

🚀 正在从本地绝对路径加载模型和 Tokenizer，这次它绝对没法联网了...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✅ 模型完美离线加载成功！当前原始层数: 36


In [5]:
print("🔪 开始执行物理剪枝...")

# Qwen 的隐藏层列表存放在 model.model.layers 中
if hasattr(model, "model") and hasattr(model.model, "layers"):
    original_layers = model.model.layers
else:
    raise ValueError("找不到 transformer layers，请检查模型结构属性名。")

# 按照我们的名单，把需要的层提取出来
pruned_layers = nn.ModuleList([original_layers[i] for i in keep_indices])

# 偷天换日：用新的层列表替换掉旧的
model.model.layers = pruned_layers

# 极其重要的一步：修改 config 文件里的层数记录！
# 如果不改这个，后续推理或训练时底层代码会报错
model.config.num_hidden_layers = len(keep_indices)

print(f"✅ 剪枝手术完成！")
print(f"现在的模型层数: {len(model.model.layers)}")
print(f"Config 中的层数: {model.config.num_hidden_layers}")

🔪 开始执行物理剪枝...
✅ 剪枝手术完成！
现在的模型层数: 18
Config 中的层数: 18


In [6]:
print(f"💾 正在将剪枝后的 4B 模型保存至: {TARGET_MODEL_PATH}")

# 确保目标文件夹存在
os.makedirs(TARGET_MODEL_PATH, exist_ok=True)

# 保存模型权重和配置文件
model.save_pretrained(TARGET_MODEL_PATH)
# 保存 Tokenizer 词表文件
tokenizer.save_pretrained(TARGET_MODEL_PATH)

print("🎉 恭喜！全新的数据驱动版 4B 剪枝模型已成功生成！")

💾 正在将剪枝后的 4B 模型保存至: /home/bev/disk4/haoxuan/SpeculativeDecoding/model/qwen3-4b-pruned-data-driven


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 恭喜！全新的数据驱动版 4B 剪枝模型已成功生成！


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 你的 4B 模型路径
MODEL_PATH = "/home/bev/disk4/haoxuan/SpeculativeDecoding/model/qwen3-4b-pruned-data-driven"

# 测试推理，我们放回 GPU 上跑，这样快一点
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 正在将剪枝后的 4B 模型加载到 {device} 进行初测...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map=device,
    trust_remote_code=True
)
model.eval() # 开启推理模式
print("✅ 4B 模型加载完毕！准备迎接'脑震荡'测试...\n" + "="*50)

# 精心设计的 5 个维度测试题
test_prompts = [
    "你好，请简单介绍一下你自己。",                  # 1. 基础指令遵循与自我认知
    "法国的首都叫什么名字？",                       # 2. 浅层事实记忆 (看看底层权重还在不在)
    "计算题：15 + 27 = ?",                        # 3. 基础数学符号处理
    "Translate 'I love programming' to Chinese.", # 4. 跨语言对齐能力
    "树上有5只鸟，猎人开枪打死了一只，树上还剩几只鸟？" # 5. 简单逻辑陷阱 (大概率翻车)
]

for i, prompt in enumerate(test_prompts):
    print(f"❓ [{i+1}] 提问: {prompt}")
    
    # 构造 Qwen 的标准对话模板
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    # 生成回答
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=50, # 限制长度，看它第一反应是什么
            temperature=0.1,   # 极低温度，关掉随机性，看它最本能的输出
            do_sample=False
        )
        
    # 截取模型生成的回复部分 (剥离掉前面的 prompt)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # 打印回答
    print(f"🤖 4B 回答: {response}")
    print("-" * 50)

🚀 正在将剪枝后的 4B 模型加载到 cuda 进行初测...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ 4B 模型加载完毕！准备迎接'脑震荡'测试...
❓ [1] 提问: 你好，请简单介绍一下你自己。
🤖 4B 回答: 
。

用户说的可能的。

。用户可能的。用户可能的的。。用户可能的的。用户可能的的，。用户可能的的的用户可能的的的的的可能的的的的
--------------------------------------------------
❓ [2] 提问: 法国的首都叫什么名字？
🤖 4B 回答: ？是需要的？
是需要的？需要更明确的？比如，需要更明确的？比如，法国，？

需要更明确的？比如，，法国人，？需要更明确的？比如，，，需要
--------------------------------------------------
❓ [3] 提问: 计算题：15 + 27 = ?
🤖 4B 回答:  the user.. the calculation... the user. the the the user. the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
--------------------------------------------------
❓ [4] 提问: Translate 'I love programming' to Chinese.
🤖 4B 回答: . the user.... the user. the time. the user. the.. the user. the. the.. the. the. the. the. the. the user the. the user the. the user the user.
--------------------------------------------------
❓ [5] 提问: 树上有5只鸟，猎人开枪打死了一只，树上还剩几只鸟？
🤖 4B 回答: 。 了 （用户需要的，，用户需要的，，用户。。。。。用户。用户。用户。用户。用户。用户。。用户。。用户。。用户。。用户。。用户。。用户
-----------------------